# 04 - Evaluation and export

Consolidate metrics from notebook 03 (including walk-forward and volatility diagnostics), compare forecasts against naive/linear baselines, inspect residuals, log experiments, and report dashboard/artifact readiness.


In [ ]:
## Setup

import os
os.environ.pop("SPARK_HOME", None)

import sys, warnings, yaml, joblib, json
from pathlib import Path
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import numpy as np
import pandas as pd
import plotly.graph_objects as go

_cwd = Path.cwd().resolve()
_candidates = list(dict.fromkeys([
    _cwd,
    *_cwd.parents,
    *(path.parent.parent for path in _cwd.glob("*/configs/pipeline.yaml")),
]))
_required_project_files = (
    "configs/pipeline.yaml",
    "src/features/pipeline.py",
    "notebooks/01_data_pipeline.ipynb",
    "Home.py",
)
_project_roots = [
    path for path in _candidates
    if all((path / relative).is_file() for relative in _required_project_files)
]
if not _project_roots:
    raise FileNotFoundError("Could not identify this project. Open its repository root or set the notebook working directory to it.")
if len(_project_roots) > 1:
    raise RuntimeError(f"Multiple project roots found: {_project_roots}. Open the intended repository root.")
PROJECT_ROOT = _project_roots[0]
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.experiment_tracker import log_run

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml', encoding='utf-8') as f:
    CFG = yaml.safe_load(f)

PROCESSED = PROJECT_ROOT / CFG['artifacts']['processed_dir']
MODELS    = PROJECT_ROOT / CFG['artifacts']['models_dir']
FIGURES   = PROJECT_ROOT / CFG['reports']['figures_dir']
METRICS   = PROJECT_ROOT / CFG['reports']['metrics_dir']
METRICS.mkdir(parents=True, exist_ok=True)

with open(PROCESSED / 'split_meta.yaml', encoding='utf-8') as f:
    SPLIT = yaml.safe_load(f)

FEATURE_COLS = SPLIT['feature_cols']
TARGET_COL   = SPLIT['target_col']
TS_COL       = SPLIT['ts_col']
TRAIN_END    = SPLIT['train_end']
VAL_END      = SPLIT['val_end']
TEST_END     = SPLIT['test_end']

print('Setup complete.')


---
## 1 - Consolidated metrics (primary: MAE/RMSE, direction, IC, MASE, naive-scaled; diagnostic: sMAPE)


In [ ]:
# Load metrics collected by notebook 03
with open(PROCESSED / 'metrics_draft.yaml', encoding='utf-8') as f:
    results_raw = yaml.safe_load(f) or []

results_df = pd.DataFrame(results_raw)
if results_df.empty:
    raise ValueError('No metrics found in metrics_draft.yaml. Run notebook 03 first.')

sort_cols = [c for c in ['partition', 'mae', 'rmse'] if c in results_df.columns]
results_df = results_df.sort_values(sort_cols).reset_index(drop=True)
metrics_csv = METRICS / 'forecast_metrics.csv'
results_df.to_csv(metrics_csv, index=False)
print(f'Metrics saved -> {metrics_csv}')
print(results_df[[c for c in ['model','partition','mae','rmse','directional_accuracy','correlation','mase','naive_scaled_mae','smape','qlike'] if c in results_df.columns]].to_string(index=False))


In [ ]:
# Test-set comparison vs naive/linear (MAE/RMSE)
test_res = results_df[results_df.get('partition') == 'Test'].copy() if 'partition' in results_df.columns else pd.DataFrame()
if not test_res.empty and 'mae' in test_res.columns:
    fig = go.Figure()
    for metric, col in [('mae', '#4FC3F7'), ('rmse', '#F7931A')]:
        if metric in test_res.columns:
            fig.add_trace(go.Bar(x=test_res['model'], y=test_res[metric], name=metric.upper(), marker_color=col))
    fig.update_layout(barmode='group', title='Test-Set Metrics Comparison (lower is better)', template='plotly_dark', yaxis_title='Error')
    fig.write_html(str(FIGURES / 'metrics_comparison.html'), include_plotlyjs='cdn')

    display_cols = [c for c in ['model','mae','rmse','directional_accuracy','correlation','mase','naive_scaled_mae','smape'] if c in test_res.columns]
    ranked = test_res[display_cols].sort_values('mae')
    print(ranked.to_string(index=False))
    if 'naive_scaled_mae' in test_res.columns:
        better = test_res[pd.to_numeric(test_res['naive_scaled_mae'], errors='coerce') < 1.0]
        if better.empty:
            print('\nNote: no model beat naive on naive-scaled MAE in this split; prefer walk-forward stability and risk/regime insights over point-return alpha.')
        else:
            print('\nModels beating naive (naive-scaled MAE < 1):', ', '.join(better.sort_values('naive_scaled_mae')['model'].astype(str).tolist()))
else:
    print('No test partition rows found.')


In [ ]:
# Walk-forward summary (if exported by notebook 03)
wf_path = METRICS / 'walk_forward_summary.json'
if wf_path.exists():
    wf = json.loads(wf_path.read_text(encoding='utf-8'))
    print('Walk-forward config:', {k: wf.get(k) for k in ['initial_train_size','test_size','step_size','expanding']})
    for name in ['ridge', 'naive_proxy']:
        agg = (wf.get(name) or {}).get('aggregate') or {}
        if agg:
            print(name, 'aggregate:', {k: agg.get(k) for k in ['mae','rmse','directional_accuracy','correlation','mase','naive_scaled_mae','n_folds'] if k in agg})
else:
    print('No walk-forward summary exported (run notebook 03).')


---
## 2 - Forecast diagnostics (residuals vs naive context)


In [ ]:
# Residual analysis for best available tree model by test MAE
MODEL_ARTIFACTS = {"LightGBM": "lgbm_return.pkl", "CatBoost": "catboost_return.pkl"}
tree_results = test_res[test_res['model'].isin(MODEL_ARTIFACTS)].sort_values('mae') if not test_res.empty else pd.DataFrame()
if not tree_results.empty:
    best = tree_results.iloc[0]
    pred_csv = MODELS / f"{'lgbm' if best['model']=='LightGBM' else 'catboost'}_test_predictions.csv"
    if pred_csv.exists():
        pred_df = pd.read_csv(pred_csv)
        if 'timestamp' in pred_df.columns:
            pred_df['timestamp'] = pd.to_datetime(pred_df['timestamp'], utc=True, errors='coerce')
        actual = pd.to_numeric(pred_df.get('actual'), errors='coerce')
        predicted = pd.to_numeric(pred_df.get('predicted'), errors='coerce')
        naive = pd.to_numeric(pred_df.get('naive'), errors='coerce') if 'naive' in pred_df.columns else None
        residuals = actual - predicted
        print(f"Residual model: {best['model']} (test MAE {best['mae']:.6f})")
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=pred_df.get('timestamp', np.arange(len(residuals))), y=residuals, mode='lines', name='Actual - predicted'))
        fig.add_trace(go.Scatter(x=pred_df.get('timestamp', np.arange(len(residuals))), y=[0]*len(residuals), name='Zero', line=dict(color='#aaa', dash='dot')))
        fig.update_layout(title=f"{best['model']} - Test Residuals", template='plotly_dark')
        fig.write_html(str(FIGURES / 'best_tree_residuals.html'), include_plotlyjs='cdn')
        fig2 = go.Figure(go.Histogram(x=residuals.dropna(), nbinsx=100, histnorm='probability density'))
        fig2.update_layout(title=f"{best['model']} - Residual Distribution", template='plotly_dark')
        fig2.write_html(str(FIGURES / 'best_tree_residual_dist.html'), include_plotlyjs='cdn')
        if naive is not None and naive.notna().any():
            bench_mae = float(np.nanmean(np.abs(actual - naive)))
            print(f'Naive MAE on same exported rows: {bench_mae:.6f}')
        print('Saved residual diagnostics.')
    else:
        print('No exported test prediction CSV found for best tree model; rerun notebook 03.')
else:
    print('No LightGBM/CatBoost test results available for residual analysis.')


---
## 3 - Regime and volatility diagnostics exports


In [ ]:
# Summarize regime diagnostics and volatility comparison if present
reg_diag_path = METRICS / 'regime_diagnostics.json'
if reg_diag_path.exists():
    reg_diag = json.loads(reg_diag_path.read_text(encoding='utf-8'))
    print(json.dumps(reg_diag, indent=2))
else:
    print('No regime diagnostics exported.')

garch_cmp = MODELS / 'garch_realized_comparison.csv'
if garch_cmp.exists():
    gdf = pd.read_csv(garch_cmp)
    print(f'GARCH vs realized-proxy rows: {len(gdf)} -> {garch_cmp}')
else:
    print('No GARCH realized comparison export found.')


---
## 4 - Experiment logging


In [ ]:
# Log test-partition runs to configs/experiments.yaml (idempotent)
logged = 0
for row in results_raw:
    if row.get('partition') != 'Test':
        continue
    model = row.get('model')
    artifact_name = MODEL_ARTIFACTS.get(model)
    artifact_p = str(MODELS / artifact_name) if artifact_name else 'n/a'
    metrics = {k: row[k] for k in ('mae','rmse','smape','directional_accuracy','correlation','mase','naive_scaled_mae') if k in row and row[k] is not None}
    log_run(
        model=model,
        target=TARGET_COL,
        train_end=TRAIN_END,
        val_end=VAL_END,
        test_end=TEST_END,
        metrics=metrics,
        artifact_path=artifact_p if artifact_p != 'n/a' and Path(artifact_p).is_file() else 'n/a',
        notes='walk-forward/diagnostic workflow; sMAPE diagnostic only for near-zero returns' if 'sMAPE' or True else '',
    )
    logged += 1
print(f'Processed {logged} test runs for experiment logging.')


---
## 5 - Dashboard readiness check


In [ ]:
# Dashboard artifact readiness (required + high-value optional)
required_for_dashboard = [PROCESSED / 'btc_features.parquet']
optional_for_dashboard = [
    MODELS / 'lgbm_return.pkl', MODELS / 'catboost_return.pkl', MODELS / 'ridge_baseline.pkl',
    MODELS / 'gmm.pkl', MODELS / 'hmm.pkl',
    MODELS / 'isolation_forest.pkl', MODELS / 'anomaly_scores.csv',
    MODELS / 'changepoints.csv', MODELS / 'garch_cond_vol.csv', MODELS / 'garch_realized_comparison.csv',
    MODELS / 'matrix_profile.csv', MODELS / 'matrix_profile_meta.json',
    MODELS / 'lgbm_test_predictions.csv', MODELS / 'catboost_test_predictions.csv',
    METRICS / 'forecast_metrics.csv', METRICS / 'walk_forward_summary.json', METRICS / 'regime_diagnostics.json',
]
missing = [p for p in required_for_dashboard if not p.exists()]
print('Required:')
for p in required_for_dashboard:
    print(f"{'READY' if p.exists() else 'MISSING'}  {p.relative_to(PROJECT_ROOT)}")
print('Optional/high-value:')
for p in optional_for_dashboard:
    print(f"{'available' if p.exists() else 'not generated'}  {p.relative_to(PROJECT_ROOT)}")
if missing:
    raise FileNotFoundError(f'Missing required dashboard data: {missing}')
print('Launch the dashboard with: streamlit run Home.py')


In [ ]:
# Final summary
print('=' * 60)
print('  PIPELINE COMPLETE (evaluation/export)')
print('=' * 60)
print(f'  Processed data : {PROCESSED}')
print(f'  Model artifacts: {MODELS}')
print(f'  Figures        : {FIGURES}')
print(f'  Metrics        : {METRICS}')
if not test_res.empty and 'mae' in test_res.columns:
    cols = [c for c in ['model','mae','rmse','directional_accuracy','mase','naive_scaled_mae'] if c in test_res.columns]
    print('\nTop test-set results by MAE (interpret vs naive/linear and walk-forward spread):')
    print(test_res.sort_values('mae').head(5)[cols].to_string(index=False))
